In [0]:
%sql
SELECT * FROM ECOMMERCE.SILVER.CLICKSTREAM LIMIT 10;

In [0]:
# %sql
# WITH BASELINE_METRICS AS (
#   SELECT
#     PRODUCT_ID,
#     EVENT_TYPE,
#     COUNT(EVENT_ID) AS EVENT_COUNT
#   FROM
#     ECOMMERCE.SILVER.clickstream
#   GROUP BY
#     1,
#     2
# )
# SELECT
#   A.PRODUCT_ID,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'PRODUCT_VIEW' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   ) AS PRODUCT_VIEW,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'ADD_TO_CART' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   ) AS ADD_TO_CART_COUNT,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'CHECKOUT' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   ) AS CHECKOUT_COUNT,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'PURCHASE' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   ) AS PURCHASE_COUNT,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'PRODUCT_VIEW' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   )
#     / SUM(
#       CASE
#         WHEN A.EVENT_TYPE = 'ADD_TO_CART' THEN A.EVENT_COUNT
#         ELSE 0
#       END
#     ) AS VIEW_TO_CART_RATE,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'ADD_TO_CART' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   )
#     / NULLIF(
#       SUM(
#         CASE
#           WHEN A.EVENT_TYPE = 'PURCHASE' THEN A.EVENT_COUNT
#           ELSE 0
#         END
#       ),
#       0
#     ) AS CART_TO_PURCHASE_RATE,
#   SUM(
#     CASE
#       WHEN A.EVENT_TYPE = 'PURCHASE' THEN A.EVENT_COUNT
#       ELSE 0
#     END
#   )
#     / NULLIF(SUM(A.EVENT_COUNT), 0) AS PURCHASE_RATE
# FROM
#   baseline_metrics A
# GROUP BY
#   1;

In [0]:
from pyspark.sql import functions as F




In [0]:
clickstream = spark.table("ECOMMERCE.SILVER.clickstream")


In [0]:
baseline_metrics = clickstream.groupBy("PRODUCT_ID", "EVENT_TYPE").agg(
    F.count("EVENT_ID").alias("EVENT_COUNT")
)

In [0]:
product_metrics = baseline_metrics.groupBy("PRODUCT_ID").agg(
    F.sum(
        F.when(F.col("EVENT_TYPE") == "PRODUCT_VIEW", F.col("EVENT_COUNT")).otherwise(0)
    ).alias("PRODUCT_VIEW"),
    F.sum(
        F.when(F.col("EVENT_TYPE") == "ADD_TO_CART", F.col("EVENT_COUNT")).otherwise(0)
    ).alias("ADD_TO_CART_COUNT"),
    F.sum(
        F.when(F.col("EVENT_TYPE") == "CHECKOUT", F.col("EVENT_COUNT")).otherwise(0)
    ).alias("CHECKOUT_COUNT"),
    F.sum(
        F.when(F.col("EVENT_TYPE") == "PURCHASE", F.col("EVENT_COUNT")).otherwise(0)
    ).alias("PURCHASE_COUNT"),
    F.sum(F.col("EVENT_COUNT")).alias("TOTAL_EVENTS"),
)

In [0]:
product_metrics = (
    product_metrics.withColumn(
        "VIEW_TO_CART_RATE",
        F.col("PRODUCT_VIEW")
        / F.when(F.col("ADD_TO_CART_COUNT") != 0, F.col("ADD_TO_CART_COUNT")),
    )
    .withColumn(
        "CART_TO_PURCHASE_RATE",
        F.col("ADD_TO_CART_COUNT")
        / F.when(F.col("PURCHASE_COUNT") != 0, F.col("PURCHASE_COUNT")),
    )
    .withColumn(
        "PURCHASE_RATE",
        F.col("PURCHASE_COUNT")
        / F.when(F.col("TOTAL_EVENTS") != 0, F.col("TOTAL_EVENTS")),
    )
    .drop("TOTAL_EVENTS")
)

In [0]:
display(product_metrics)

In [0]:
product_metrics.write.format("delta").mode("APPEND").saveAsTable(
    "ecommerce.gold.product_conversion"
)

In [0]:
view_median = product_metrics.approxQuantile("PRODUCT_VIEW", [0.50], 0.01)[0]

purchase_rate_median = product_metrics.approxQuantile("PURCHASE_RATE", [0.50], 0.01)[0]

high_interest_low_conversion = product_metrics.filter(
    (F.col("PRODUCT_VIEW") >= view_median)
    & (F.col("PURCHASE_RATE") < purchase_rate_median)
).orderBy(F.col("PRODUCT_VIEW").desc(), F.col("PURCHASE_RATE").asc())

display(high_interest_low_conversion)